In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_squared_error
from sklearn.base import clone

import sys
import os

import joblib

sys.path.append(os.path.abspath(".."))

from src.models import LINEAR_MODELS, TREE_MODELS

ENSEMBLE_NUM = 2

In [3]:
category_cols = ["category", "day_of_week"]
target_col = "late_duration_min"

feature_df = pd.read_parquet("../data/feature_df.parquet")
feature_df

,day_of_week,distance_km,category,late_duration_min
0,4,1.274747,lunch,7.0
1,1,1.773078,exercise,35.0
2,3,15.205063,dinner/drinks,48.0
3,4,1.130494,dinner/drinks,60.0
4,6,1.274747,breakfast,10.0
5,5,15.077114,work/career fair,30.0
6,4,15.077114,dinner/drinks,9.0
7,4,1.274747,lunch,13.0
8,2,5.656470,lunch,17.0


In [4]:
def Cat_LabelEncoding(df, cols):
    modified_df = df.copy()

    le = LabelEncoder()

    for col in cols:
        modified_df[col] = le.fit_transform(modified_df[col])

    modified_df.drop(columns=cols)

    return modified_df

def Cat_OneHotEncoding(df, cols):
    modified_df = pd.get_dummies(df, columns=["category", "day_of_week"])

    return modified_df

In [5]:
# Split features and target
y = feature_df[target_col]

X_label = Cat_LabelEncoding(feature_df.drop(columns=[target_col]), category_cols)
X_onehot = Cat_OneHotEncoding(feature_df.drop(columns=[target_col]), category_cols)


In [6]:
def loocv_mse(model, X, y):
    loo = LeaveOneOut()
    errors = []

    for train_idx, test_idx in loo.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model_clone = clone(model)
        model_clone.fit(X_train, y_train)
        pred = model_clone.predict(X_test)

        errors.append(mean_squared_error(y_test, pred))

    return np.mean(errors)

In [7]:
results = {}

# Linear models (one-hot encoding)
for name, model in LINEAR_MODELS:
    mse = loocv_mse(model, X_onehot, y)
    results[name] = {
        "mse": mse,
        "model": model,
        "type": "linear"
    }

# Tree models (label encoding)
for name, model in TREE_MODELS:
    mse = loocv_mse(model, X_label, y)
    results[name] = {
        "mse": mse,
        "model": model,
        "type": "tree"
    }

# Rank models
sorted_results = sorted(results.items(), key=lambda x: x[1]["mse"])

for name, info in sorted_results:
    print(f"{name}: MSE = {info['mse']:.4f}")

random_forest: MSE = 588.8447
gboost: MSE = 651.5241
ridge: MSE = 713.1707
linear_regression: MSE = 1062.9486


In [8]:
# Select top N models based on LOOCV results
top_models = [name for name, _ in sorted_results[:ENSEMBLE_NUM]]

# Retrain each selected model using ALL available data
trained_models = {}

for name, info in results.items():
    # Create a fresh copy of the model
    model = clone(info["model"])

    # Choose correct feature representation
    # - Linear models → one-hot encoded features
    # - Tree models   → label encoded features
    if info["type"] == "linear":
        model.fit(X_onehot, y)
    else:
        model.fit(X_label, y)

    # Store trained model + metadata
    trained_models[name] = {
        "model": model,
        "type": info["type"],
        "mse": info["mse"]
    }

trained_models

{'linear_regression': {'model': LinearRegression(),
  'type': 'linear',
  'mse': np.float64(1062.9485807185604)},
 'ridge': {'model': Ridge(),
  'type': 'linear',
  'mse': np.float64(713.1707297277426)},
 'random_forest': {'model': RandomForestRegressor(),
  'type': 'tree',
  'mse': np.float64(588.8447054444446)},
 'gboost': {'model': GradientBoostingRegressor(),
  'type': 'tree',
  'mse': np.float64(651.5241436759363)}}

In [9]:
def ensemble_predict(trained_models, model_names, X_df):
    preds = []

    for name in model_names:
        model_info = trained_models[name]

        if model_info["type"] == "linear":
            preds.append(
                model_info["model"].predict(
                    Cat_OneHotEncoding(X_df.drop(columns=[target_col]), category_cols)
                )
            )
        else:
            preds.append(
                model_info["model"].predict(
                    Cat_LabelEncoding(X_df.drop(columns=[target_col]), category_cols)
                )
            )

    return np.mean(preds, axis=0)

final_pred = ensemble_predict(
    trained_models,
    top_models,
    feature_df
)

final_pred

array([11.25788334, 32.15101357, 41.83342732, 52.57764096, 11.73196559,
       26.72898063, 13.44899862, 11.25788334, 19.07870663])

In [10]:
result_df = feature_df.copy()
result_df[f"pred_{target_col}"] = final_pred
result_df["models_used"] = ", ".join(top_models)

result_df

,day_of_week,distance_km,category,late_duration_min,pred_late_duration_min,models_used
0,4,1.274747,lunch,7.0,11.257883,"random_forest, gboost"
1,1,1.773078,exercise,35.0,32.151014,"random_forest, gboost"
2,3,15.205063,dinner/drinks,48.0,41.833427,"random_forest, gboost"
3,4,1.130494,dinner/drinks,60.0,52.577641,"random_forest, gboost"
4,6,1.274747,breakfast,10.0,11.731966,"random_forest, gboost"
5,5,15.077114,work/career fair,30.0,26.728981,"random_forest, gboost"
6,4,15.077114,dinner/drinks,9.0,13.448999,"random_forest, gboost"
7,4,1.274747,lunch,13.0,11.257883,"random_forest, gboost"
8,2,5.656470,lunch,17.0,19.078707,"random_forest, gboost"


### Save Model Object

In [11]:
os.makedirs("../models", exist_ok=True)

joblib.dump(trained_models, "../models/trained_models.pkl")
joblib.dump(top_models, "../models/top_models.pkl")

['../models/top_models.pkl']